In [1]:
import os
from pathlib import Path

import duckdb
from dotenv import load_dotenv

root = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
load_dotenv(root / ".env", override=True)

mart_path = (
    "s3://mart/seoul-bike-sharing-demand/"
    "source_ingested_on=2026-09-30/hourly_rentals.parquet"
)

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

endpoint = os.environ["S3_ENDPOINT"].removeprefix("http://").removeprefix("https://")
con.execute("SET s3_endpoint = ?", [endpoint])
con.execute("SET s3_access_key_id = ?", [os.environ["S3_ACCESS_KEY"]])
con.execute("SET s3_secret_access_key = ?", [os.environ["S3_SECRET_KEY"]])
con.execute("SET s3_url_style = 'path'")
con.execute("SET s3_use_ssl = false")

print("Витрина:", mart_path)

Витрина: s3://mart/seoul-bike-sharing-demand/source_ingested_on=2026-09-30/hourly_rentals.parquet


In [2]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE hourly_rentals AS
    SELECT event_date, hour_of_day, rented_bike_count, temperature_c, functioning_day
    FROM read_parquet(?, hive_partitioning = false)
""", [mart_path])

con.execute("""
    CREATE OR REPLACE TEMP VIEW working_hours AS
    SELECT
        event_date,
        hour_of_day,
        rented_bike_count,
        temperature_c,
        CASE
            WHEN MONTH(event_date) IN (12, 1, 2) THEN 'Winter'
            WHEN MONTH(event_date) IN (3, 4, 5) THEN 'Spring'
            WHEN MONTH(event_date) IN (6, 7, 8) THEN 'Summer'
            WHEN MONTH(event_date) IN (9, 10, 11) THEN 'Autumn'
        END AS season
    FROM hourly_rentals
    WHERE functioning_day = 'Yes'
""")

con.sql("""
    SELECT
        MIN(event_date) AS first_date,
        MAX(event_date) AS last_date,
        COUNT(*) AS mart_rows,
        COUNT(*) FILTER (WHERE functioning_day = 'Yes') AS working_rows,
        COUNT(*) FILTER (WHERE functioning_day = 'No') AS non_working_rows
    FROM hourly_rentals
""").show(max_width=120)

┌────────────┬────────────┬───────────┬──────────────┬──────────────────┐
│ first_date │ last_date  │ mart_rows │ working_rows │ non_working_rows │
│    date    │    date    │   int64   │    int64     │      int64       │
├────────────┼────────────┼───────────┼──────────────┼──────────────────┤
│ 2017-12-01 │ 2018-11-30 │      8760 │         8465 │              295 │
└────────────┴────────────┴───────────┴──────────────┴──────────────────┘



In [3]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE metrics_total AS
    SELECT
        COUNT(*) AS observations,
        COUNT(rented_bike_count) AS rentals_used,
        REGR_COUNT(rented_bike_count, temperature_c) AS pairs_used,
        AVG(rented_bike_count) AS avg_rentals,
        CORR(rented_bike_count, temperature_c) AS temp_rental_corr
    FROM working_hours
""")

con.sql("""
    SELECT
        observations, rentals_used, pairs_used,
        ROUND(avg_rentals, 2) AS avg_rentals,
        ROUND(temp_rental_corr, 4) AS temp_rental_corr
    FROM metrics_total
""").show(max_width=120)

┌──────────────┬──────────────┬────────────┬─────────────┬──────────────────┐
│ observations │ rentals_used │ pairs_used │ avg_rentals │ temp_rental_corr │
│    int64     │    int64     │   uint32   │   double    │      double      │
├──────────────┼──────────────┼────────────┼─────────────┼──────────────────┤
│         8465 │         8465 │       8465 │      729.16 │           0.5627 │
└──────────────┴──────────────┴────────────┴─────────────┴──────────────────┘



In [4]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE metrics_by_hour AS
    WITH by_hour AS (
        SELECT
            hour_of_day,
            COUNT(*) AS observations,
            COUNT(rented_bike_count) AS rentals_used,
            REGR_COUNT(rented_bike_count, temperature_c) AS pairs_used,
            AVG(rented_bike_count) AS avg_rentals,
            CORR(rented_bike_count, temperature_c) AS temp_rental_corr
        FROM working_hours
        GROUP BY hour_of_day
    )
    SELECT
        *,
        100.0 * observations / NULLIF(SUM(observations) OVER (), 0) AS share_pct
    FROM by_hour
""")

con.sql("""
    SELECT
        hour_of_day, observations, rentals_used, pairs_used,
        ROUND(avg_rentals, 2) AS avg_rentals,
        ROUND(temp_rental_corr, 4) AS temp_rental_corr,
        ROUND(share_pct, 3) AS share_pct
    FROM metrics_by_hour
    ORDER BY hour_of_day
""").show(max_rows=24, max_width=130)

┌─────────────┬──────────────┬──────────────┬────────────┬─────────────┬──────────────────┬───────────┐
│ hour_of_day │ observations │ rentals_used │ pairs_used │ avg_rentals │ temp_rental_corr │ share_pct │
│    int32    │    int64     │    int64     │   uint32   │   double    │      double      │  double   │
├─────────────┼──────────────┼──────────────┼────────────┼─────────────┼──────────────────┼───────────┤
│           0 │          352 │          352 │        352 │      561.46 │           0.7914 │     4.158 │
│           1 │          352 │          352 │        352 │      441.92 │           0.7355 │     4.158 │
│           2 │          352 │          352 │        352 │      312.77 │            0.693 │     4.158 │
│           3 │          352 │          352 │        352 │      210.84 │           0.6952 │     4.158 │
│           4 │          352 │          352 │        352 │      137.49 │           0.7195 │     4.158 │
│           5 │          352 │          352 │        352 │      

In [5]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE metrics_by_hour_season AS
    SELECT
        hour_of_day,
        season,
        COUNT(*) AS observations,
        COUNT(rented_bike_count) AS rentals_used,
        REGR_COUNT(rented_bike_count, temperature_c) AS pairs_used,
        AVG(rented_bike_count) AS avg_rentals,
        CORR(rented_bike_count, temperature_c) AS temp_rental_corr
    FROM working_hours
    GROUP BY hour_of_day, season
""")

con.sql("""
    SELECT
        hour_of_day, season, observations, rentals_used, pairs_used,
        ROUND(avg_rentals, 2) AS avg_rentals,
        ROUND(temp_rental_corr, 4) AS temp_rental_corr
    FROM metrics_by_hour_season
    WHERE hour_of_day IN (0, 8, 14)
    ORDER BY hour_of_day, season
""").show(max_width=130)

┌─────────────┬─────────┬──────────────┬──────────────┬────────────┬─────────────┬──────────────────┐
│ hour_of_day │ season  │ observations │ rentals_used │ pairs_used │ avg_rentals │ temp_rental_corr │
│    int32    │ varchar │    int64     │    int64     │   uint32   │   double    │      double      │
├─────────────┼─────────┼──────────────┼──────────────┼────────────┼─────────────┼──────────────────┤
│           0 │ Autumn  │           80 │           80 │         80 │      709.44 │           0.7066 │
│           0 │ Spring  │           90 │           90 │         90 │      481.09 │           0.6095 │
│           0 │ Summer  │           92 │           92 │         92 │      899.07 │           -0.016 │
│           0 │ Winter  │           90 │           90 │         90 │      165.18 │            0.563 │
│           8 │ Autumn  │           81 │           81 │         81 │     1345.04 │          -0.0248 │
│           8 │ Spring  │           90 │           90 │         90 │     1036.38 │

In [6]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE temperature_rental_pairs AS
    SELECT event_date, hour_of_day, season, temperature_c, rented_bike_count
    FROM working_hours
    WHERE temperature_c IS NOT NULL
      AND rented_bike_count IS NOT NULL
""")

con.sql("""
    SELECT *
    FROM temperature_rental_pairs
    ORDER BY event_date, hour_of_day
    LIMIT 5
""").show(max_width=120)

┌────────────┬─────────────┬─────────┬───────────────┬───────────────────┐
│ event_date │ hour_of_day │ season  │ temperature_c │ rented_bike_count │
│    date    │    int32    │ varchar │    double     │       int64       │
├────────────┼─────────────┼─────────┼───────────────┼───────────────────┤
│ 2017-12-01 │           0 │ Winter  │          -5.2 │               254 │
│ 2017-12-01 │           1 │ Winter  │          -5.5 │               204 │
│ 2017-12-01 │           2 │ Winter  │          -6.0 │               173 │
│ 2017-12-01 │           3 │ Winter  │          -6.2 │               107 │
│ 2017-12-01 │           4 │ Winter  │          -6.0 │                78 │
└────────────┴─────────────┴─────────┴───────────────┴───────────────────┘



In [7]:
groups_check = con.sql("""
    SELECT
        (SELECT observations FROM metrics_total) AS total_rows,
        (SELECT SUM(observations) FROM metrics_by_hour) AS hourly_rows,
        (SELECT SUM(observations) FROM metrics_by_hour_season) AS seasonal_rows,
        (SELECT COUNT(*) FROM metrics_by_hour) AS hour_groups,
        (SELECT COUNT(*) FROM metrics_by_hour_season) AS season_groups,
        (SELECT SUM(share_pct) FROM metrics_by_hour) AS share_total
""")

groups_check.show(max_width=130)
total, hourly, seasonal, hours, hour_seasons, share = groups_check.fetchone()

assert total == hourly == seasonal > 0
assert hours == 24 and hour_seasons == 96
assert abs(share - 100) < 1e-8

┌────────────┬─────────────┬───────────────┬─────────────┬───────────────┬───────────────────┐
│ total_rows │ hourly_rows │ seasonal_rows │ hour_groups │ season_groups │    share_total    │
│   int64    │   int128    │    int128     │    int64    │     int64     │      double       │
├────────────┼─────────────┼───────────────┼─────────────┼───────────────┼───────────────────┤
│       8465 │        8465 │          8465 │          24 │            96 │ 99.99999999999997 │
└────────────┴─────────────┴───────────────┴─────────────┴───────────────┴───────────────────┘



In [8]:
counts_check = con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE rented_bike_count IS NOT NULL) AS eligible_rentals,
        (SELECT rentals_used FROM metrics_total) AS rentals_used,
        COUNT(*) FILTER (
            WHERE rented_bike_count IS NOT NULL AND temperature_c IS NOT NULL
        ) AS eligible_pairs,
        (SELECT pairs_used FROM metrics_total) AS pairs_used,
        (SELECT COUNT(*) FROM temperature_rental_pairs) AS sample_rows
    FROM working_hours
""")

counts_check.show(max_width=130)
eligible_rentals, rentals_used, eligible_pairs, pairs_used, sample_rows = counts_check.fetchone()

assert eligible_rentals == rentals_used
assert eligible_pairs == pairs_used == sample_rows

┌──────────────────┬──────────────┬────────────────┬────────────┬─────────────┐
│ eligible_rentals │ rentals_used │ eligible_pairs │ pairs_used │ sample_rows │
│      int64       │    int64     │     int64      │   uint32   │    int64    │
├──────────────────┼──────────────┼────────────────┼────────────┼─────────────┤
│             8465 │         8465 │           8465 │       8465 │        8465 │
└──────────────────┴──────────────┴────────────────┴────────────┴─────────────┘

